In [88]:
import torch
import lightning.pytorch as pl

In [242]:
# x = torch.randn((3, 3))
x = torch.randint(-15, 15, (3,3))
print(x)

tensor([[  1,  10,  -7],
        [ 13,   7,   9],
        [  0,   3, -14]])


## BReLU 기본형

In [89]:
class BReLU(pl.LightningModule):
    def __init__(self):
        super(BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        return x * epsilon.to(self.device)

In [109]:
brelu = BReLU()
brelu(x)

tensor([[ 0., 10.,  1.],
        [-0., 14., -0.],
        [-2.,  5.,  4.]])

In [122]:
print(brelu)

BReLU()


## Leaky BReLU

In [110]:
class Leaky_BReLU(pl.LightningModule):
    def __init__(self):
        super(Leaky_BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = torch.where(epsilon==0, 0.1, 1.0)
        
        return x * epsilon.to(self.device)

In [111]:
leaky = Leaky_BReLU()
leaky(x)

tensor([[ 0.0000, 10.0000,  1.0000],
        [-0.9000, 14.0000, -1.0000],
        [-2.0000,  5.0000,  4.0000]])

In [123]:
print(leaky)

Leaky_BReLU()


## 점근 속도를 조절하는 BReLU

In [124]:
class VariableBReLU(pl.LightningModule):
    def __init__(self, alpha=1):
        super(VariableBReLU, self).__init__()
        self.alpha = alpha
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=self.alpha * x).sample()
        
        return x * epsilon.to(self.device)
    
    def extra_repr(self) -> str:
        return 'alpha={}'.format(self.alpha)

In [134]:
def mv(**kwargs):
    if 'alpha' in list(kwargs.keys()):
        return VariableBReLU(alpha=kwargs.get('alpha'))

mv(alpha=.1)

VariableBReLU(alpha=0.1)

In [125]:
alpha = 0.1
vb = VariableBReLU(alpha=alpha)
vb(x)

tensor([[  0.,  10.,   0.],
        [ -0.,  14., -10.],
        [ -2.,   5.,   0.]])

In [126]:
print(vb)

VariableBReLU(alpha=0.1)


## 결과값을 0아니면 1로 만들기

In [137]:
class MeasureAct(pl.LightningModule):
    def __init__(self):
        super(MeasureAct, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = epsilon * (1/(x + 1e-16))
        
        return x * epsilon.to(self.device)

In [138]:
ma = MeasureAct()
ma(x)

tensor([[0., 1., 1.],
        [0., 1., 0.],
        [0., 1., 1.]])

## 일부 원소만 0 아니면 1로 만들기

In [175]:
class SomeMeasureAct(pl.LightningModule):
    def __init__(self):
        super(SomeMeasureAct, self).__init__()
        
    def forward(self, x):
        b = torch.distributions.bernoulli.Bernoulli(logits=x)
        b1 = b.sample()
        b2 = b.sample()

        epsilon = b1*b2*(1/(x+1e-16)) + torch.logical_xor(b1, b2)
        
        return x * epsilon

In [188]:
sma = SomeMeasureAct()
sma(x)

tensor([[0., 1., 1.],
        [-0., 1., -0.],
        [-0., 1., 4.]])

## 확률적으로 ReLU, BReLU를 적용하기

In [263]:
class SARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)
        
    # def extra_repr(self) -> str:
    #     return 'zero={}, one={}'.format(self.zero, self.one)
        

In [264]:
sarb = SARB()
sarb(x)

tensor([[ 1, 10,  0],
        [13,  7,  9],
        [ 0,  3,  0]])

In [265]:
sarb

SARB(
  (zero): BReLU()
  (one): ReLU(inplace=True)
)

## 확률 식으로 sin, cos 사용

In [278]:
class BwithSinCos(pl.LightningModule):
    def __init__(self):
        super(BwithSinCos, self).__init__()
        
    def forward(self, x):
        p = -torch.sin(x) * torch.cos(x + (torch.pi/2))
        print(p)
        epsilon = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        return x * epsilon.to(self.device)

In [284]:
sc = BwithSinCos()
sc(x)

tensor([[0.7081, 0.2960, 0.0000],
        [0.1765, 0.4316, 0.1698],
        [0.0000, 0.0199, 0.0000]])


tensor([[ 0.,  0.,  0.],
        [13.,  7.,  0.],
        [ 0.,  0.,  0.]])

## 얽힘 (구현 방법 생각 중)

모든 칸을 돌면서 자기 주위의 1 개수를 센 다음 % 2 한 값을 자기 위치에 갖는다.

In [289]:
b = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
o = torch.zeros_like(b)
print(b)
print(o)

tensor([[0., 1., 0.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [315]:
a = torch.randint(0, 2, (3, 4, 5))
a

tensor([[[0, 1, 1, 1, 0],
         [0, 1, 1, 0, 0],
         [0, 0, 0, 1, 1],
         [0, 0, 1, 1, 1]],

        [[0, 1, 0, 1, 1],
         [0, 0, 0, 0, 0],
         [0, 0, 1, 0, 1],
         [1, 1, 0, 1, 0]],

        [[0, 0, 1, 0, 1],
         [0, 0, 0, 1, 0],
         [1, 0, 0, 1, 0],
         [1, 0, 0, 0, 0]]])

In [348]:
(a==1).nonzero(as_tuple=True)
a[(a==1).nonzero(as_tuple=True)] += 1 

In [349]:
a

tensor([[[0, 2, 2, 2, 0],
         [0, 2, 2, 0, 0],
         [0, 0, 0, 2, 2],
         [0, 0, 2, 2, 2]],

        [[0, 2, 0, 2, 2],
         [0, 0, 0, 0, 0],
         [0, 0, 2, 0, 2],
         [2, 2, 0, 2, 0]],

        [[0, 0, 2, 0, 2],
         [0, 0, 0, 2, 0],
         [2, 0, 0, 2, 0],
         [2, 0, 0, 0, 0]]])

In [344]:
ones = (b==1).nonzero(as_tuple=False)

for i in ones:
    print(i)
    print(b[tuple(i)])

tensor([0, 1])
tensor(1.)
tensor([1, 0])
tensor(1.)
tensor([1, 1])
tensor(1.)
tensor([1, 2])
tensor(1.)
tensor([2, 0])
tensor(1.)
tensor([2, 1])
tensor(1.)
tensor([2, 2])
tensor(1.)


-------------------------------------

In [71]:
f = torch.distributions.bernoulli.Bernoulli(logits=alpha * x)

In [72]:
f.probs

tensor([[0.3543, 0.5744, 0.4013],
        [0.3543, 0.2891, 0.5000],
        [0.5744, 0.2142, 0.1978]])

In [73]:
e = torch.exp(alpha * x) / (torch.exp(alpha * x) + 1)
print(e)

tensor([[0.3543, 0.5744, 0.4013],
        [0.3543, 0.2891, 0.5000],
        [0.5744, 0.2142, 0.1978]])


In [86]:
p = -torch.sin(x) * torch.cos(x+ (torch.pi/2))
print(p)
torch.distributions.bernoulli.Bernoulli(probs=p).sample()

tensor([[0.0781, 0.0199, 0.5727],
        [0.0781, 0.1698, 0.0000],
        [0.0199, 0.1765, 0.9813]])


tensor([[0., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])